# Forward-mode AD from scratch

*Adapted from:* https://michael-herbst.com/talks/2025.10.09_IPAM_DFT_Gradients_1_ad.html

We will provide a simplistic, but functional forward-mode AD implementation. The basic idea is to compute the value and derivatives synchronously, by overloading every primitive operation.

We define a new number type for doing that:

In [1]:
struct Dual <: Number
    x::Float64   # Value
    δx::Float64  # Derivative
end

This number type we now equip with the basic differentiation rules:

In [2]:
begin
    # (f+g)'(x) = f'(x) + g'(x) 
    Base.:+(a::Dual, b::Dual) = Dual(a.x + b.x, a.δx + b.δx)
    Base.:-(a::Dual, b::Dual) = error("TODO missing rule") # TODO implement rule

    # (f*g)'(x) = f(x)*g'(x) + f'(x)*g(x)
    Base.:*(a::Dual, b::Dual) = error("TODO missing rule") # TODO implement rule
    Base.:/(a::Dual, b::Dual) = Dual(a.x / b.x, (b.x * a.δx - a.x  * b.δx) / b.x^2)
end

In [3]:
# # Solution
#
# begin
#     # (f+g)'(x) = f'(x) + g'(x) 
#     Base.:+(a::Dual, b::Dual) = Dual(a.x + b.x, a.δx + b.δx)
#     Base.:-(a::Dual, b::Dual) = Dual(a.x - b.x, a.δx - b.δx)

#     # (f*g)'(x) = f(x)*g'(x) + f'(x)*g(x)
#     Base.:*(a::Dual, b::Dual) = Dual(a.x * b.x, a.x * b.δx + a.δx * b.x )
#     Base.:/(a::Dual, b::Dual) = Dual(a.x / b.x, (b.x * a.δx - a.x  * b.δx) / b.x^2)
# end

On top of this we need to tell julia, that any number is also a dual number, just with an empty derivative:



In [4]:
begin
    Base.convert(::Type{Dual}, x::Real) = Dual(x, zero(x))
    Base.promote_rule(::Type{Dual}, ::Type{<:Number}) = Dual
end

A derivative is now obtained by starting with a unit derivative and the desired value and just propagating through. We try to compute
$$
\left. \frac{d}{dx} \left(  x^3 - x^2 \right) \right|_{x=2} = 3 \cdot 2^2 - 2\cdot2 = 8
$$
We introduce:

In [5]:
value_and_derivative(f, x::Number) = f(  Dual(x, one(x))  )

value_and_derivative (generic function with 1 method)

and compute:

In [6]:
value_and_derivative(x -> x^3 - x^2, 2.0)

LoadError: TODO missing rule

---

Now we have understood the basic implementation idea of forward-mode AD by dual numbers,
and what it means to define **custom rules** of differentiation.

In practice, we rely on https://github.com/JuliaDiff/ForwardDiff.jl for dual numbers,
and in [DFTK.jl](https://github.com/JuliaMolSim/DFTK.jl) we define a **few additional custom rules** for
- handling external calls outside of Julia (FFTW for fast Fourier transforms, spglib for symmetry detection)
- handling `self_consistent_field` (by triggering DFPT solver) $\implies$ **AD-DFPT**